In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
import torch.nn as nn
print("Files Python can see:", os.listdir('.'))
from google.colab import drive # type: ignore
drive.mount(r'/content/drive/')

# Important Parameters
learning_rate = 0.1
epochs = 25

# Load Data
df = pd.read_csv(r"/content/drive/MyDrive/DL_CSV/breast-cancer.csv")

# FIXED: Saved the dataframe modification back to df so 'id' is removed
df = df.drop(['id'], axis='columns', errors='ignore')

Y = df['diagnosis']
X = df.drop(['diagnosis'], axis='columns')

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Label Encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Converting Numpy Arrays to tensors
#Keep in mind the data types, self.linear = nn.Linear(num_features, 1) creates weights with dtype torch.float32 by default and the dataset was of datatype double
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1).float()

class Model(nn.Module):

    def __init__(self, num_features):
        super().__init__() #invoking constructor of the parent class

        self.linear=nn.Linear(num_features,1) #Format(number_of_input, numberof_output)
        self.sigmoid=nn.Sigmoid()

    def forward(self,features):
        out=self.linear(features) #calculate wx+b
        out=self.sigmoid(out)
        return out

model=Model(X_train_tensor.shape[1])

#Adding Loss Function
loss_function=nn.BCELoss()

optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

#Implementing Dataset and DataLoader Class
from torch.utils.data import Dataset,DataLoader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features=features
        self.labels=labels  

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index],self.labels[index]
    
#Making object of CustomDataset
train_dataset=CustomDataset(X_train_tensor,y_train_tensor)
test_dataset=CustomDataset(X_test_tensor,y_test_tensor)
print(len(test_dataset))

#DataLoader class
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2)
test_dataloader=DataLoader(test_dataset,batch_size=32,shuffle=True)

#Training the model on the batches made
for i in range(epochs):

    for batch_features,batch_labels in train_dataloader:

        y_pred=model(batch_features)

        loss=loss_function(y_pred,batch_labels.view(-1,1))

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_dataloader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')


Mounted at /content/drive/
114
Accuracy: 0.5669
